In [0]:
%pip install scikit-learn
%pip install xgboost
%pip install matplotlib

from numpy import absolute
from pandas import read_csv
from sklearn.model_selection import cross_val_score, train_test_split, RepeatedKFold
from xgboost import XGBRegressor
import matplotlib.pyplot as plt

In [0]:
import pandas as pd

df_air_quality = spark.table("workspace.mlfsbook.lugano_aq_fg").toPandas()
df_air_quality.sort_values(by=["date"], ascending=False).head()

In [0]:
import numpy as np

df_weather = spark.table("workspace.mlfsbook.lugano_weather_fg").toPandas()

# Normalize dates

df_weather['date'] = pd.to_datetime(df_weather['date']).dt.normalize()

# Encode wind_direction_10m_dominant (degrees) as its sine value
df_weather['dominant_wind_direction_sin'] = np.sin(np.deg2rad(df_weather['wind_direction_10m_dominant']))

df_weather.sort_values(by=["date"], ascending=False).head()

In [0]:
df_dataset = pd.merge(df_air_quality, df_weather, on="date", how="right")

df_dataset.info()

In [0]:
df_dataset.head()

In [0]:
# Remove forecast rows
df_training = df_dataset.dropna(subset=['pm25'])

# Add day of week as a feature
df_training = df_training.assign(day_of_week=df_training['date'].dt.dayofweek)
feature_cols = ["temperature_2m_mean", "precipitation_sum", "wind_speed_10m_max", "dominant_wind_direction_sin", "day_of_week"]
df_training = df_training[feature_cols + ["pm25", "date"]]

# Split into train/test sets: test set is only January 2026
df_training['date'] = pd.to_datetime(df_training['date'])

display(df_training.head(10))

In [0]:
test_mask = (df_training['date'].dt.year == 2026) & (df_training['date'].dt.month == 1)
train_mask = (df_training['date'].dt.year == 2025)
df_test = df_training[test_mask]
df_train = df_training[train_mask]

X_train = df_train[feature_cols].to_numpy()
y_train = df_train["pm25"].to_numpy()
X_test = df_test[feature_cols].to_numpy()
y_test = df_test["pm25"].to_numpy()

# Grid search for XGBoost hyperparameters (max 10 combinations)
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# MSE (Mean Squared Error) measures average squared difference between predicted and true values (lower is better).
# R-squared measures proportion of variance explained by the model (closer to 1 is better).

param_grid = {
    'n_estimators': [50, 100],
    'learning_rate': [0.05, 0.1, 0.2]
}
grid = list(ParameterGrid(param_grid))  # Limit to 10 combinations

mse_scores = []
r2_scores = []
params_list = []

# Run with default XGBRegressor parameters
model_default = XGBRegressor()
model_default.fit(X_train, y_train)
y_pred_default = model_default.predict(X_test)
mse_default = mean_squared_error(y_test, y_pred_default)
r2_default = r2_score(y_test, y_pred_default)
mse_scores.append(mse_default)
r2_scores.append(r2_default)
params_list.append("default")

# Run grid search
for params in grid:
    model = XGBRegressor(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mse_scores.append(mse)
    r2_scores.append(r2)
    params_list.append(str(params))

# Create a DataFrame for visualization
import pandas as pd
results_df = pd.DataFrame({
    'Params': params_list,
    'MSE': mse_scores,
    'R2': r2_scores
})

# Pick the best model (lowest MSE)
best_idx = np.argmin(mse_scores)
best_params = params_list[best_idx]

if best_params == "default":
    model = XGBRegressor()
else:
    # Convert string to dict for params
    import ast
    model = XGBRegressor(**ast.literal_eval(best_params))

# Fit best model on all available data

model.fit(X_train, y_train)

display(results_df)

In [0]:
# Feature importance plot
importances = model.feature_importances_
plt.figure(figsize=(8, 4))
plt.barh(feature_cols, importances)
plt.xlabel("Feature Importance")
plt.title("XGBoost Feature Importance")
plt.tight_layout()
fig_feature_importance = plt.gcf()
plt.show()

In [0]:
from matplotlib.ticker import ScalarFormatter
from matplotlib.patches import Patch

df_test = df_test.drop_duplicates(subset=['date'])

# Predictions on test set
df_test = df_test.assign(pm25_pred=model.predict(df_test[feature_cols].to_numpy()))

plt.figure(figsize=(12, 6))
ax = plt.gca()

colors = ['green', 'yellow', 'orange', 'red', 'purple', 'darkred']
labels = ['Good', 'Moderate', 'Unhealthy for Some', 'Unhealthy', 'Very Unhealthy', 'Hazardous']
ranges = [(0, 49), (50, 99), (100, 149), (150, 199), (200, 299), (300, 500)]

for color, (start, end) in zip(colors, ranges):
    ax.axhspan(start, end, color=color, alpha=0.3)

ax.plot(df_test['date'], df_test['pm25'], label='Actual PM2.5', marker='o')
ax.plot(df_test['date'], df_test['pm25_pred'], label='Predicted PM2.5', marker='o')
ax.set_yscale('log')
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.set_xlabel('Date')
ax.set_ylabel('PM2.5 (log scale)')
ax.set_title('Actual vs Predicted PM2.5 (Log Scale)')

patches = [Patch(color=colors[i], label=f"{labels[i]}: {ranges[i][0]}-{ranges[i][1]}") for i in range(len(colors))]
legend1 = ax.legend(handles=patches, loc='upper right', title="Air Quality Categories", fontsize='x-small')
ax.legend(loc='upper left')

plt.tight_layout()

fig_jan_2026_predictions = plt.gcf()

plt.show()

In [0]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

# Infer signature from training data and predictions
signature = infer_signature(
    X_train,  # your training features
    model.predict(X_train)  # model predictions on training features
)

with mlflow.start_run():
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        registered_model_name="workspace.mlfsbook.PM25_XGBoost_Model_Lugano",
        signature=signature
    )
    mlflow.log_metrics({
        "mse": mse_scores[best_idx],
        "r2": r2_scores[best_idx]
    })
    fig_feature_importance.savefig("feature_importance.png")
    mlflow.log_artifact("feature_importance.png", artifact_path="plots")
    fig_jan_2026_predictions.savefig("test_set_predictions.png")
    mlflow.log_artifact("test_set_predictions.png", artifact_path="plots")